# Study 954 — High Yield in Disguise — the teardown

The constrained held-out replication, R² by return horizon, the excess-of-cash Sharpe race, the vol-matched Newey-West *t*, block-bootstrap CIs, the era cut, the crisis table, the cost and estimation-window sweeps, the two cross-check funds, and a live **synthetic** control.

**Construction.** For each day *t*, `w` is the OLS slope of `r_HY − r_IEF` on `r_SPY − r_IEF` over the trailing 252 trading days — the constraint that weights sum to one is imposed by subtracting the duration leg from both sides, so `w` is literally the equity share of a fully funded blend. `w` is frozen at each **calendar month-end** and applied to the **following** month: that freeze is the study's **single execution lag** (the weight in force on day *t* was fitted on returns ending at least one trading day earlier, and is never re-fitted intra-month). No second lag is stacked on it.

**Frictions.** The replication pays 2 bps one-way × NAV on rebalance turnover (a PROXY, swept) and 50 bps/yr borrow on any short leg (a PROXY that is inert — the fitted `w` never left [0, 1]). The fund pays **nothing**: it is bought once and held, so the race is deliberately generous to it. Both arms are excess-of-cash (BIL total return), both tapes are **total return** (`auto_adjust=True`).

All real numbers are frozen from `docs/results.md` (Fingerprint `c08a8d88f0f9`, as-of 2026-06-30).

In [1]:
R = {'stamp_start': '2007-05-30', 'stamp_end': '2026-06-30', 'stamp_n': 4802, 'fp': 'c08a8d88f0f9', 'start': '2008-06-02', 'end': '2026-06-30', 'n_days': 4548, 'w_mean': 0.45, 'w_min': 0.269, 'w_max': 0.631, 'short_max': 0.0, 'turnover': 0.34, 'r2_d': 0.468, 'r2_w': 0.565, 'r2_m': 0.517, 'r2_q': 0.591, 'te_d': 8.15, 'te_w': 7.45, 'te_m': 7.18, 'te_q': 6.26, 'resid_ann': -2.16, 't_resid': -1.25, 'hy_sharpe': 0.395, 'hy_cagr': 3.85, 'hy_cagr_lived': 5.17, 'hy_vol': 11.12, 'hy_dd': -32.4, 'hy_t': 1.73, 'rp_sharpe': 0.731, 'rp_cagr': 6.34, 'rp_cagr_lived': 7.68, 'rp_vol': 8.96, 'rp_dd': -20.7, 'rp_t': 3.39, 'cash_cagr_lived': 1.26, 'gap': -0.336, 't_gap': -2.07, 'gap_pp_per_yr': 3.7, 'ci_hy_lo': -0.049, 'ci_hy_hi': 0.9, 'ci_hy_neg': 4.4, 'ci_rp_lo': 0.314, 'ci_rp_hi': 1.205, 'ci_rp_neg': 0.1, 'ci_gap_pt': -0.427, 'ci_gap_lo': -0.805, 'ci_gap_hi': -0.035, 'ci_gap_neg': 98.4, 'era_e_n': 2163, 'era_e_hy': 0.48, 'era_e_rp': 0.77, 'era_e_gap': -0.29, 'era_e_t': -1.19, 'era_e_resid': -1.07, 'era_e_tres': -0.34, 'era_l_n': 2385, 'era_l_hy': 0.298, 'era_l_rp': 0.696, 'era_l_gap': -0.398, 'era_l_t': -2.05, 'era_l_resid': -3.15, 'era_l_tres': -1.97, 'c08_dd_hy': -32.4, 'c08_dd_rp': -20.7, 'c08_ret_hy': -9.2, 'c08_ret_rp': -8.9, 'c20_dd_hy': -22.0, 'c20_dd_rp': -13.1, 'c20_ret_hy': -6.9, 'c20_ret_rp': 1.4, 'c22_dd_hy': -15.5, 'c22_dd_rp': -18.9, 'c22_ret_hy': -11.0, 'c22_ret_rp': -15.7, 'cost0_gap': -0.337, 'cost0_t': -2.07, 'cost25_gap': -0.328, 'cost25_t': -2.02, 'win126_gap': -0.25, 'win126_t': -1.58, 'win252_gap': -0.336, 'win252_t': -2.07, 'win504_gap': -0.373, 'win504_t': -2.32, 'win756_gap': -0.389, 'win756_t': -2.41, 'leg_shy_w': 0.357, 'leg_shy_gap': -0.297, 'leg_shy_t': -1.93, 'leg_shy_resid': -0.62, 'leg_iei_w': 0.393, 'leg_iei_gap': -0.323, 'leg_iei_t': -2.07, 'leg_iei_resid': -1.33, 'leg_ief_w': 0.45, 'leg_ief_gap': -0.336, 'leg_ief_t': -2.07, 'leg_ief_resid': -2.16, 'leg_tlt_w': 0.614, 'leg_tlt_gap': -0.344, 'leg_tlt_t': -1.92, 'leg_tlt_resid': -4.19, 'jnk_n': 4399, 'jnk_w': 0.443, 'jnk_r2': 0.487, 'jnk_gap': -0.268, 'jnk_t': -1.64, 'ushy_n': 1923, 'ushy_w': 0.367, 'ushy_r2': 0.63, 'ushy_gap': -0.286, 'ushy_t': -1.24, 'er_hyg': 0.49, 'er_blend': 0.125, 'er_diff': 0.365}
print('frozen real-tape headline loaded: %d fields, fingerprint %s' % (len(R), R['fp']))

frozen real-tape headline loaded: 115 fields, fingerprint c08a8d88f0f9


## 1. The held-out weight path

> 💡 **In plain words:** we are asking what mixture of stocks and government bonds behaved most like a junk-bond fund, and we only ever use yesterday's answer to judge today.

In [2]:
print(f"held-out record : {R['start']} -> {R['end']}  n={R['n_days']:,}")
print(f"data stamp      : {R['stamp_start']} -> {R['stamp_end']}  "
      f"n={R['stamp_n']:,}  fp={R['fp']}")
print(f"equity share w  : mean {R['w_mean']:.3f}  range [{R['w_min']:.3f}, {R['w_max']:.3f}]")
print(f"max short notional {R['short_max']:.3f} -> the borrow PROXY is identically inert")
print(f"turnover {R['turnover']:.2f} x NAV / yr -> the cost PROXY has almost no room to bite")

held-out record : 2008-06-02 -> 2026-06-30  n=4,548
data stamp      : 2007-05-30 -> 2026-06-30  n=4,802  fp=c08a8d88f0f9
equity share w  : mean 0.450  range [0.269, 0.631]
max short notional 0.000 -> the borrow PROXY is identically inert
turnover 0.34 x NAV / yr -> the cost PROXY has almost no room to bite


## 2. Replication quality by horizon — the 'costume' test

A fund whose bonds were marked stale would look badly replicated daily and well replicated quarterly. The R² is flat in the horizon, so the unexplained part is economic, not an artefact of marking.

In [3]:
for h, r2, te in [('daily', R['r2_d'], R['te_d']), ('weekly', R['r2_w'], R['te_w']),
                  ('monthly', R['r2_m'], R['te_m']), ('quarterly', R['r2_q'], R['te_q'])]:
    print(f"{h:>10s}: R^2 {r2:.3f}   tracking error {te:5.2f}%/yr")
print()
print(f"residual (HY - replication): {R['resid_ann']:+.2f}%/yr   HAC t = {R['t_resid']:+.2f}")
print('-> the replication FAILS as a replication: ~half the variance is credit-specific.')

     daily: R^2 0.468   tracking error  8.15%/yr
    weekly: R^2 0.565   tracking error  7.45%/yr
   monthly: R^2 0.517   tracking error  7.18%/yr
 quarterly: R^2 0.591   tracking error  6.26%/yr

residual (HY - replication): -2.16%/yr   HAC t = -1.25
-> the replication FAILS as a replication: ~half the variance is credit-specific.


## 3. The excess-of-cash race and the vol-matched HAC *t*

Both arms minus BIL's total return; drawdowns are absolute (lived). The gap is tested as the Newey-West *t* on the daily difference of the two series after each is scaled to unit realised volatility — the Jobson-Korkie Sharpe comparison in HAC form.

> 💡 **In plain words:** dial both portfolios to the same riskiness, then ask which one ended up with more money, and whether the difference is bigger than chance.

In [4]:
print(f"HYG          : exSharpe {R['hy_sharpe']:+.3f}  exCAGR {R['hy_cagr']:+.2f}%  "
      f"vol {R['hy_vol']:.2f}%  MaxDD {R['hy_dd']:.2f}%  HAC t {R['hy_t']:+.2f}")
print(f"replication  : exSharpe {R['rp_sharpe']:+.3f}  exCAGR {R['rp_cagr']:+.2f}%  "
      f"vol {R['rp_vol']:.2f}%  MaxDD {R['rp_dd']:.2f}%  HAC t {R['rp_t']:+.2f}")
print(f"lived (absolute) CAGR: HYG {R['hy_cagr_lived']:+.2f}%  "
      f"replication {R['rp_cagr_lived']:+.2f}%  cash {R['cash_cagr_lived']:+.2f}%"
      f"  <- exCAGR is net of cash; the drawdowns above are absolute")
print()
print(f"excess-Sharpe gap (HY - repl): {R['gap']:+.3f}   vol-matched HAC t = {R['t_gap']:+.2f}")
print('   (the vol match uses full-sample realised vols: an EX-POST test')
print('    statistic, not a path anyone could have levered to in advance)')
print(f"-> about {R['gap_pp_per_yr']:.1f} pp/yr of excess return forgone at matched vol")
print(f"-> of which ~{R['er_diff']:.2f} pp is the fee gap "
      f"(HYG {R['er_hyg']:.2f}% vs blend {R['er_blend']:.3f}%, a PROXY decomposition)")

HYG          : exSharpe +0.395  exCAGR +3.85%  vol 11.12%  MaxDD -32.40%  HAC t +1.73
replication  : exSharpe +0.731  exCAGR +6.34%  vol 8.96%  MaxDD -20.70%  HAC t +3.39
lived (absolute) CAGR: HYG +5.17%  replication +7.68%  cash +1.26%  <- exCAGR is net of cash; the drawdowns above are absolute

excess-Sharpe gap (HY - repl): -0.336   vol-matched HAC t = -2.07
   (the vol match uses full-sample realised vols: an EX-POST test
    statistic, not a path anyone could have levered to in advance)
-> about 3.7 pp/yr of excess return forgone at matched vol
-> of which ~0.36 pp is the fee gap (HYG 0.49% vs blend 0.125%, a PROXY decomposition)


## 4. Block-bootstrap CIs (2,000 draws, 21-day blocks)

In [5]:
print(f"HYG exSharpe        : {R['hy_sharpe']:+.3f}  95% CI "
      f"[{R['ci_hy_lo']:+.3f}, {R['ci_hy_hi']:+.3f}]  share<0 {R['ci_hy_neg']:.1f}%")
print(f"replication exSharpe: {R['rp_sharpe']:+.3f}  95% CI "
      f"[{R['ci_rp_lo']:+.3f}, {R['ci_rp_hi']:+.3f}]  share<0 {R['ci_rp_neg']:.1f}%")
print(f"vol-matched gap     : {R['ci_gap_pt']:+.3f}  95% CI "
      f"[{R['ci_gap_lo']:+.3f}, {R['ci_gap_hi']:+.3f}]  share<0 {R['ci_gap_neg']:.1f}%")
print('-> the gap CI excludes zero, but the upper end sits at -0.04. Marginal.')

HYG exSharpe        : +0.395  95% CI [-0.049, +0.900]  share<0 4.4%
replication exSharpe: +0.731  95% CI [+0.314, +1.205]  share<0 0.1%
vol-matched gap     : -0.427  95% CI [-0.805, -0.035]  share<0 98.4%
-> the gap CI excludes zero, but the upper end sits at -0.04. Marginal.


## 5. Era cut (split 2017-01-01)

Re-uses the already held-out series, so both halves inherit exactly the out-of-sample weights the full run used.

In [6]:
print(f"2008-2016 (n={R['era_e_n']:,}): HY {R['era_e_hy']:+.3f} / repl {R['era_e_rp']:+.3f}  "
      f"gap {R['era_e_gap']:+.3f} (t={R['era_e_t']:+.2f})  "
      f"residual {R['era_e_resid']:+.2f}%/yr (t={R['era_e_tres']:+.2f})")
print(f"2017-2026 (n={R['era_l_n']:,}): HY {R['era_l_hy']:+.3f} / repl {R['era_l_rp']:+.3f}  "
      f"gap {R['era_l_gap']:+.3f} (t={R['era_l_t']:+.2f})  "
      f"residual {R['era_l_resid']:+.2f}%/yr (t={R['era_l_tres']:+.2f})")
print('-> same sign in both halves, wider in the recent one; only the recent one clears |t|=2.')

2008-2016 (n=2,163): HY +0.480 / repl +0.770  gap -0.290 (t=-1.19)  residual -1.07%/yr (t=-0.34)
2017-2026 (n=2,385): HY +0.298 / repl +0.696  gap -0.398 (t=-2.05)  residual -3.15%/yr (t=-1.97)
-> same sign in both halves, wider in the recent one; only the recent one clears |t|=2.


## 6. The crisis table — where the two arms actually diverge

> 💡 **In plain words:** the blend beats the fund when the crisis is about companies defaulting, and loses when the crisis is about interest rates.

In [7]:
rows = [('2008 GFC', R['c08_dd_hy'], R['c08_dd_rp'], R['c08_ret_hy'], R['c08_ret_rp']),
        ('2020 Covid', R['c20_dd_hy'], R['c20_dd_rp'], R['c20_ret_hy'], R['c20_ret_rp']),
        ('2022 rates', R['c22_dd_hy'], R['c22_dd_rp'], R['c22_ret_hy'], R['c22_ret_rp'])]
print(f"{'episode':<12s}{'DD hy':>9s}{'DD repl':>10s}{'ret hy':>9s}{'ret repl':>10s}   winner")
for tag, ddh, ddr, rh, rr in rows:
    win = 'replication' if ddr > ddh else 'the fund'
    print(f"{tag:<12s}{ddh:>8.1f}%{ddr:>9.1f}%{rh:>8.1f}%{rr:>9.1f}%   {win}")
print()
print('2022 is a pure duration event: the blend holds far more interest-rate risk.')
print('The swap is credit risk -> duration risk, not risk -> no risk.')

episode         DD hy   DD repl   ret hy  ret repl   winner
2008 GFC       -32.4%    -20.7%    -9.2%     -8.9%   replication
2020 Covid     -22.0%    -13.1%    -6.9%      1.4%   replication
2022 rates     -15.5%    -18.9%   -11.0%    -15.7%   the fund

2022 is a pure duration event: the blend holds far more interest-rate risk.
The swap is credit risk -> duration risk, not risk -> no risk.


## 7. Sensitivity — cost PROXY and estimation window

Turnover is 0.34 ×NAV/yr, so friction cannot explain the result; the estimation window is the one design choice with real bite, and it trades weight stability against how much of 2008 survives into the out-of-sample record.

In [8]:
print('cost sweep (one-way bps on the replication):')
print(f"   0 bps: gap {R['cost0_gap']:+.3f} (t={R['cost0_t']:+.2f})")
print(f"   2 bps: gap {R['win252_gap']:+.3f} (t={R['win252_t']:+.2f})   <- headline")
print(f"  25 bps: gap {R['cost25_gap']:+.3f} (t={R['cost25_t']:+.2f})   <- 12x the headline, unchanged")
print()
print('estimation-window sweep:')
for w, g, t, s in [(126, R['win126_gap'], R['win126_t'], '2007-12'),
                   (252, R['win252_gap'], R['win252_t'], '2008-06'),
                   (504, R['win504_gap'], R['win504_t'], '2009-06'),
                   (756, R['win756_gap'], R['win756_t'], '2010-06')]:
    print(f"  {w:>3d} d (OOS from {s}): gap {g:+.3f} (t={t:+.2f})")
print('-> sign unanimous, magnitude monotone in the window, t straddles the bar.')

cost sweep (one-way bps on the replication):
   0 bps: gap -0.337 (t=-2.07)
   2 bps: gap -0.336 (t=-2.07)   <- headline
  25 bps: gap -0.328 (t=-2.02)   <- 12x the headline, unchanged

estimation-window sweep:
  126 d (OOS from 2007-12): gap -0.250 (t=-1.58)
  252 d (OOS from 2008-06): gap -0.336 (t=-2.07)
  504 d (OOS from 2009-06): gap -0.373 (t=-2.32)
  756 d (OOS from 2010-06): gap -0.389 (t=-2.41)
-> sign unanimous, magnitude monotone in the window, t straddles the bar.


## 7b. Sensitivity — which Treasury is *the* duration leg?

IEF (7-10y) is a **design choice**, not a fact of the tape: a high-yield fund's own duration is nearer 3-4 years, so SHY (1-3y) and IEI (3-7y) are at least as defensible and TLT (20y+) is the aggressive end. The weight re-fits itself to whatever leg it is handed, so the fit always works — the question is whether the *conclusion* does. Treasuries only: AGG or LQD on the bench would smuggle the credit risk under test into the benchmark. Same common 4,548-day sample.

In [9]:
print(f"{'leg':<12s}{'w':>7s}{'residual':>11s}{'gap':>9s}{'t':>8s}")
for tag, w, res, g, t in [
        ('SHY  1-3y', R['leg_shy_w'], R['leg_shy_resid'], R['leg_shy_gap'], R['leg_shy_t']),
        ('IEI  3-7y', R['leg_iei_w'], R['leg_iei_resid'], R['leg_iei_gap'], R['leg_iei_t']),
        ('IEF 7-10y', R['leg_ief_w'], R['leg_ief_resid'], R['leg_ief_gap'], R['leg_ief_t']),
        ('TLT  20y+', R['leg_tlt_w'], R['leg_tlt_resid'], R['leg_tlt_gap'], R['leg_tlt_t'])]:
    flag = '   <- headline' if tag.startswith('IEF') else ''
    print(f"{tag:<12s}{w:>7.3f}{res:>10.2f}%{g:>9.3f}{t:>8.2f}{flag}")
print()
print('-> the SIGN is leg-proof: every maturity hands the blend the higher Sharpe.')
print('-> the t is NOT: 2.07 is the top of a 1.92-2.07 range, so the bar is')
print('   cleared in the middle and missed at both ends of the curve.')
print('-> the residual is the least robust number in the study: -0.62%/yr against')
print('   SHY, -4.19%/yr against TLT. Most of the headline -2.16%/yr is the price')
print('   of the maturity mismatch, not a measurement of unpaid credit risk.')

leg               w   residual      gap       t
SHY  1-3y     0.357     -0.62%   -0.297   -1.93
IEI  3-7y     0.393     -1.33%   -0.323   -2.07
IEF 7-10y     0.450     -2.16%   -0.336   -2.07   <- headline
TLT  20y+     0.614     -4.19%   -0.344   -1.92

-> the SIGN is leg-proof: every maturity hands the blend the higher Sharpe.
-> the t is NOT: 2.07 is the top of a 1.92-2.07 range, so the bar is
   cleared in the middle and missed at both ends of the curve.
-> the residual is the least robust number in the study: -0.62%/yr against
   SHY, -4.19%/yr against TLT. Most of the headline -2.16%/yr is the price
   of the maturity mismatch, not a measurement of unpaid credit risk.


## 8. Cross-checks — the other two high-yield funds

In [10]:
print(f"JNK  (n={R['jnk_n']:,}): w {R['jnk_w']:.3f}  R^2 {R['jnk_r2']:.3f}  "
      f"gap {R['jnk_gap']:+.3f} (t={R['jnk_t']:+.2f})")
print(f"USHY (n={R['ushy_n']:,}): w {R['ushy_w']:.3f}  R^2 {R['ushy_r2']:.3f}  "
      f"gap {R['ushy_gap']:+.3f} (t={R['ushy_t']:+.2f})")
print('-> same sign and size, but neither clears |t|=2 on its own shorter sample.')
print('Survivorship: these are the funds still listed; dead HY ETFs are absent,')
print('which flatters the fund side - so the negative finding is conservative.')

JNK  (n=4,399): w 0.443  R^2 0.487  gap -0.268 (t=-1.64)
USHY (n=1,923): w 0.367  R^2 0.630  gap -0.286 (t=-1.24)
-> same sign and size, but neither clears |t|=2 on its own shorter sample.
Survivorship: these are the funds still listed; dead HY ETFs are absent,
which flatters the fund side - so the negative finding is conservative.


## 9. Live synthetic control — the machinery is unbiased

**This section is synthetic, not the real tape.** The generator builds a fund that *is* `w_true × equity + (1 − w_true) × duration` plus an idiosyncratic credit shock of fixed size; `signal_strength` changes only what that shock is *paid*. At `1` it carries a 3%/yr give-up (the planted effect); at `0` it earns exactly the premium that keeps the fund's Sharpe level with the blend's (the null). The weight must be recovered in **both** worlds.

In [11]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from hy_replication import data, strategy as st

for ss, tag in [(1.0, 'uncompensated (planted)'), (0.0, 'fairly paid (null)  ')]:
    prices, truth = data.synthetic_panel(signal_strength=ss, seed=954)
    d = st.synthetic_detect(prices)
    print(f"{tag}: w_hat {d['w_mean']:.3f} (true {truth['w_true']:.2f})  "
          f"R^2 {d['r2']:.3f}  residual {d['residual_ann']*100:+.2f}%/yr "
          f"(t={d['t_residual']:+.2f})  gap {d['excess_sharpe_gap']:+.3f} "
          f"(t={d['t_gap']:+.2f})")

uncompensated (planted): w_hat 0.456 (true 0.45)  R^2 0.653  residual -3.79%/yr (t=-2.65)  gap -0.511 (t=-3.56)


fairly paid (null)  : w_hat 0.456 (true 0.45)  R^2 0.653  residual +0.15%/yr (t=+0.11)  gap -0.126 (t=-0.88)


In [12]:
for ss, tag in [(1.0, 'planted'), (0.0, 'null   ')]:
    gaps = np.array([
        st.synthetic_detect(data.synthetic_panel(signal_strength=ss, seed=954 + s)[0])['excess_sharpe_gap']
        for s in range(8)
    ])
    print(f"{tag} x8 seeds: gap mean {gaps.mean():+.3f} (sd {gaps.std(ddof=1):.3f}), "
          f"|gap| >= 0.35 on {(abs(gaps) >= 0.35).sum()}/8")
print()
print('SYNTHETIC ONLY. The detector fires on the planted give-up and is centred on')
print('zero when the same extra risk is fairly paid -> the real-tape gap is a fact')
print('about the high-yield tape, not a biased harness.')

planted x8 seeds: gap mean -0.437 (sd 0.129), |gap| >= 0.35 on 5/8


null    x8 seeds: gap mean -0.050 (sd 0.128), |gap| >= 0.35 on 0/8

SYNTHETIC ONLY. The detector fires on the planted give-up and is centred on
zero when the same extra risk is fairly paid -> the real-tape gap is a fact
about the high-yield tape, not a biased harness.


## Verdict

- **Signal — Mixed.** Two claims, two answers. *Replication* is **refuted**: R² = 0.468 daily and 0.591 quarterly with ~7 pp/yr of tracking error, so high yield carries a real, distinct credit exposure and is not a repackaged equity position. *Compensation* leans one way everywhere — the gap is negative in all four estimation windows (-0.250 to -0.389), both eras (-0.290 / -0.398) and all three funds — but the headline HAC *t* is only -2.07, the bootstrap CI [-0.805, -0.035] clears zero by 0.04, the standalone residual *t* is -1.25, JNK (-1.64), USHY (-1.24) and the early era (-1.19) all miss the bar, and swapping the duration leg for SHY (-1.93) or TLT (-1.92) drops the headline back under it. Unanimous direction, marginal size — Mixed, not Real.
- **Tradability — Fragile.** The substitution is cheap and mechanical (0.34 ×NAV/yr, unchanged at 25 bps, `w` inside [0, 1] so no leverage or borrow) and it delivered +2.5 pp/yr of lived CAGR with a 11.7 pp shallower worst loss. But it is a *substitution*, not an alpha — you keep the same two risk premia and drop a fund fee worth 0.36 pp — and it is regime-conditional: 2022 cost the blend 4.7 pp more than the fund, because the trade is credit risk for duration risk. Size it as an exposure decision, not as an edge.